# 원하는 포즈로 이미지 만드는 도구

N017 "이미지 생성 어플리케이션 만들기" 챕터 8 과제.

**하는 일**: 참조 사진에서 사람의 관절 위치(포즈)만 뽑아서, 그 자세는 그대로 유지한 채 완전히 새로운 이미지를 생성합니다.
이것이 **ControlNet의 Pose 조건**입니다 — 프롬프트만으로는 정확한 자세를 재현하기 어렵지만, 참조 사진의 뼈대 좌표를 강제로 고정하면 자세가 그대로 유지됩니다.

**사용 모델**: Stable Diffusion 1.5 + `lllyasviel/sd-controlnet-openpose` (ControlNet을 처음 만든 곳에서 공개한 검증된 조합, 무료 T4 GPU에서 여유 있게 동작)

**실행 순서**: 이 노트북 상단 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU` 로 바꾼 뒤, 셀을 위에서부터 순서대로 실행하세요.

## 1단계 — 필요한 라이브러리 설치

`diffusers`(이미지 생성 파이프라인), `controlnet_aux`(포즈 추출 도구), `accelerate`를 설치합니다. 몇 분 걸릴 수 있습니다.

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux opencv-python-headless

## 2단계 — 포즈 참조 이미지 업로드

아래 셀을 실행하면 파일 선택 창이 뜹니다. `samples` 폴더에 있는 **pose_01.jpg**와 **pose_02.jpg** 두 장을 함께 선택해서 업로드하세요.

In [ ]:
from google.colab import files
import os

os.makedirs("samples", exist_ok=True)
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, os.path.join("samples", fname))
print("업로드된 파일:", os.listdir("samples"))

## 3단계 — 포즈 추출 (OpenPose)

사진에서 사람 관절 위치만 뽑아 스켈레톤(뼈대) 이미지로 바꿉니다. 이 스켈레톤이 ControlNet에게 "이 자세를 유지해줘"라고 알려주는 조건이 됩니다.

In [ ]:
from controlnet_aux import OpenposeDetector
from diffusers.utils import load_image
import matplotlib.pyplot as plt

openpose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

pose_paths = ["samples/pose_01.jpg", "samples/pose_02.jpg"]
ref_images = [load_image(p) for p in pose_paths]
pose_skeletons = [openpose(img) for img in ref_images]

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i in range(2):
    axes[i][0].imshow(ref_images[i]); axes[i][0].set_title(f"원본 pose_0{i+1}"); axes[i][0].axis("off")
    axes[i][1].imshow(pose_skeletons[i]); axes[i][1].set_title(f"추출된 스켈레톤"); axes[i][1].axis("off")
plt.tight_layout(); plt.show()

## 4단계 — ControlNet + Stable Diffusion 파이프라인 불러오기

최초 1회 모델을 내려받습니다(약 5GB, 1~2분 소요). guidance_scale은 기본 모델용으로 7.5를 씁니다(N017 강의에서 배운 것처럼, 증류 모델이 아니므로 0.0이 아니라 이 값을 씁니다).

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose", torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
print("파이프라인 준비 완료")

## 5단계 — 프롬프트 작성 (6칸 원칙)

N017 강의에서 배운 6칸(피사체·동작·카메라·조명·환경·스타일)에 맞춰 프롬프트를 씁니다. 동작(자세)은 이미 ControlNet이 고정하므로, 나머지 다섯 칸(피사체·카메라·조명·환경·스타일)을 채우는 데 집중하면 됩니다.

아래 두 프롬프트를 원하는 대로 바꿔보세요. `PROMPT_1`은 pose_01(나무 자세)에, `PROMPT_2`는 pose_02(삼각 자세)에 적용됩니다.

In [ ]:
SEED = 42

PROMPT_1 = "a warrior in bronze armor standing on a mountain cliff at sunrise, golden hour lighting, cinematic photography, epic fantasy style"
PROMPT_2 = "a dancer in a flowing red silk dress on a rainy city street at night, neon lights reflecting on wet pavement, cinematic photography"

NEGATIVE = "low quality, blurry, bad anatomy, extra limbs, watermark"

## 6단계 — 이미지 생성 + 결과 저장

각 포즈마다 이미지를 생성하고, `samples/output_01.png` / `samples/output_02.png`로 저장합니다. 나중에 재현할 수 있도록 프롬프트·시드·모델 이름도 함께 JSON으로 저장합니다.

In [ ]:
import json

prompts = [PROMPT_1, PROMPT_2]
results = []

for i, (skeleton, prompt) in enumerate(zip(pose_skeletons, prompts), start=1):
    generator = torch.Generator(device="cuda").manual_seed(SEED)
    image = pipe(
        prompt,
        negative_prompt=NEGATIVE,
        image=skeleton,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]

    out_path = f"samples/output_0{i}.png"
    image.save(out_path)
    meta = {"prompt": prompt, "negative_prompt": NEGATIVE, "seed": SEED,
            "model": "runwayml/stable-diffusion-v1-5 + lllyasviel/sd-controlnet-openpose"}
    with open(f"samples/output_0{i}.json", "w") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    results.append(image)
    print(f"저장됨: {out_path}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for i, img in enumerate(results):
    axes[i].imshow(img); axes[i].set_title(f"output_0{i+1}"); axes[i].axis("off")
plt.tight_layout(); plt.show()

## 7단계 — 결과 다운로드

생성된 이미지를 내 컴퓨터로 내려받습니다. 이 파일들을 로컬 `pose-image-tool/samples/` 폴더에 넣으면 과제 제출 준비가 끝납니다.

In [ ]:
from google.colab import files as colab_files

for i in range(1, 3):
    colab_files.download(f"samples/output_0{i}.png")
    colab_files.download(f"samples/output_0{i}.json")